# Model selection on the validation period

This notebook applies Section 4.4 of the evaluation protocol. Every ARIMA and Random Forest candidate forecasts the validation period (2022–2023) one session ahead, and the candidate with the lowest validation RMSE in each family is selected.

**The test period is not used here.** The series is cut at the end of 2023 immediately after loading, so no code below can reach 2024–2025.

Running the whole notebook takes roughly 20 minutes on a recent desktop, most of it in the Random Forest candidates.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# The notebook lives in notebooks/, one level below the project root.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from vixforecast.data import SPLITS, load_vix
from vixforecast.evaluation import mae, rmse, walk_forward
from vixforecast.models import Arima, Naive, forest, har

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

## 1. Data

In [2]:
START, END = SPLITS["validation"]

closes = load_vix().loc[:END]  # the test period is dropped here
actual = closes.loc[START:END]

print(f"Closes available: {closes.index[0].date()} to {closes.index[-1].date()} ({len(closes):,})")
print(f"Validation targets: {actual.index[0].date()} to {actual.index[-1].date()} ({len(actual)})")

Closes available: 2015-01-02 to 2023-12-29 (2,264)
Validation targets: 2022-01-03 to 2023-12-29 (501)


## 2. Walk-forward evaluation

`walk_forward` re-estimates a model before the first forecast of each calendar month, using every close before that date, and keeps the parameters fixed for the rest of the month. Each forecast sees only closes dated before its target.

In [3]:
def evaluate(model):
    """Forecast every validation date and return the two error measures."""
    forecasts = walk_forward(closes, model, START, END)
    return {"RMSE": rmse(actual, forecasts), "MAE": mae(actual, forecasts)}

## 3. Reference forecasts

The naive forecast and the HAR regression have nothing to select. Their validation errors are the reference points for the tables below.

In [4]:
reference = pd.DataFrame({"Naive": evaluate(Naive()), "HAR": evaluate(har())}).T
naive_rmse = reference.loc["Naive", "RMSE"]
reference.round(4)

,RMSE,MAE
Naive,1.4279,1.0335
HAR,1.4220,1.0105


## 4. ARIMA candidates

Eighteen candidates: p and q from 0 to 2, d of 0 or 1. Candidates with d = 0 include a constant; candidates with d = 1 do not.

In [5]:
started = time.perf_counter()
rows = []
for d in (0, 1):
    for p in (0, 1, 2):
        for q in (0, 1, 2):
            model = Arima(p, d, q)
            try:
                scores = evaluate(model)
            except RuntimeError as error:  # first estimation did not converge: candidate dropped
                print(error)
                continue
            rows.append({
                "p": p, "d": d, "q": q,
                "parameters": p + q + (1 if d == 0 else 0),
                **scores,
                "failed re-estimations": len(model.failed_fits),
            })

arima_table = pd.DataFrame(rows)
arima_table["RMSE / naive"] = arima_table["RMSE"] / naive_rmse
print(f"{len(arima_table)} candidates evaluated in {(time.perf_counter() - started) / 60:.1f} minutes")
arima_table.sort_values("RMSE").round(4)

18 candidates evaluated in 2.9 minutes


,p,d,q,parameters,RMSE,MAE,failed re-estimations,RMSE / naive
3,1,0,0,2,1.4221,1.0124,0,0.9959
9,0,1,0,0,1.4279,1.0335,0,1.0000
4,1,0,1,3,1.4313,1.0163,0,1.0024
6,2,0,0,3,1.4366,1.0205,0,1.0061
10,0,1,1,1,1.4371,1.0345,0,1.0064
7,2,0,1,4,1.4374,1.0208,0,1.0066
12,1,1,0,1,1.4426,1.0384,0,1.0103
13,1,1,1,2,1.4432,1.0390,0,1.0107
15,2,1,0,2,1.4437,1.0394,0,1.0110
5,1,0,2,4,1.4444,1.0258,0,1.0116


In [6]:
# Implementation check from the protocol: ARIMA(0,1,0) without a constant is the naive forecast.
check = arima_table.query("p == 0 and d == 1 and q == 0")["RMSE"].iloc[0]
assert np.isclose(check, naive_rmse, atol=1e-6), "ARIMA(0,1,0) does not reproduce the naive forecast."
print(f"ARIMA(0,1,0) RMSE {check:.6f} = naive RMSE {naive_rmse:.6f}")

ARIMA(0,1,0) RMSE 1.427907 = naive RMSE 1.427907


**Reading.** All 18 candidates were estimated without a failed re-estimation, so none was dropped.

- ARIMA(1,0,0) has the lowest RMSE (1.4221), 0.4% below the naive forecast. It is the only candidate that beats naive on RMSE.
- Among the differenced candidates (d = 1), the best is ARIMA(0,1,0), which is the naive forecast itself. Every additional AR or MA term makes the forecast slightly worse.
- The six candidates with d = 0 and an autoregressive term all have a lower MAE than naive (by 0.7% to 2.0%), although only one has a lower RMSE.
- The three candidates with d = 0 and no autoregressive term are far worse (1.9 to 4.6 times the naive RMSE). Without an AR term they cannot follow a series this persistent.

Fifteen of the eighteen candidates are within 2.4% of the naive RMSE. The choice between them rests on very small differences.

## 5. Random Forest candidates

Twenty-four candidates: two targets, two feature sets, three minimum leaf sizes and two split settings. All use 500 trees and the same random seed.

In [7]:
started = time.perf_counter()
rows = []
for target in ("level", "change"):
    for features in ("har", "extended"):
        for min_samples_leaf in (5, 20, 50):
            for split_features in ("third", "all"):
                model = forest(target, features, min_samples_leaf, split_features)
                rows.append({
                    "target": target, "features": features,
                    "min_samples_leaf": min_samples_leaf, "split_features": split_features,
                    **evaluate(model),
                })

forest_table = pd.DataFrame(rows)
forest_table["RMSE / naive"] = forest_table["RMSE"] / naive_rmse
print(f"{len(forest_table)} candidates evaluated in {(time.perf_counter() - started) / 60:.1f} minutes")
forest_table.sort_values("RMSE").round(4)

24 candidates evaluated in 18.5 minutes


,target,features,min_samples_leaf,split_features,RMSE,MAE,RMSE / naive
16,change,har,50,third,1.4193,1.0214,0.9940
17,change,har,50,all,1.4286,1.0350,1.0005
14,change,har,20,third,1.4305,1.0334,1.0018
22,change,extended,50,third,1.4313,1.0336,1.0023
23,change,extended,50,all,1.4422,1.0424,1.0100
20,change,extended,20,third,1.4446,1.0392,1.0117
15,change,har,20,all,1.4566,1.0540,1.0201
12,change,har,5,third,1.4589,1.0655,1.0217
21,change,extended,20,all,1.4658,1.0534,1.0265
18,change,extended,5,third,1.4693,1.0520,1.0290


**Reading.**

- **Target.** Predicting the change is clearly better than predicting the level. The twelve change-target candidates range from 0.6% better to 7.1% worse than naive; all twelve level-target candidates are worse than naive, by 4.6% to 48%.
- **Leaf size.** For the change target, larger leaves are better. Daily changes are mostly noise, so heavy averaging helps.
- **Features.** The extended feature set did not improve on the three HAR inputs: its best candidate is 0.2% worse than naive.
- **Selected candidate.** Change target, HAR inputs, at least 50 observations per leaf, one feature considered per split. Its RMSE of 1.4193 is 0.6% below naive, and it is the only one of the 24 that beats naive on RMSE.

## 6. Selection

The rule is fixed in the protocol: lowest validation RMSE within each family. Ties to four decimal places go to the simpler candidate, meaning fewer ARIMA parameters or the larger minimum leaf size.

In [8]:
def select(table, tie_break, ascending):
    """Return the row with the lowest RMSE rounded to four decimals, then the simpler candidate."""
    ranked = table.assign(rounded=table["RMSE"].round(4))
    ranked = ranked.sort_values(["rounded", tie_break], ascending=[True, ascending], kind="stable")
    return ranked.iloc[0]

best_arima = select(arima_table, "parameters", ascending=True)
best_forest = select(forest_table, "min_samples_leaf", ascending=False)

selected = {
    "selected_on": f"validation period, {actual.index[0].date()} to {actual.index[-1].date()}",
    "criterion": "lowest validation RMSE",
    "arima": {"p": int(best_arima["p"]), "d": int(best_arima["d"]), "q": int(best_arima["q"])},
    "random_forest": {
        "target": best_forest["target"],
        "features": best_forest["features"],
        "min_samples_leaf": int(best_forest["min_samples_leaf"]),
        "split_features": best_forest["split_features"],
    },
}
print(json.dumps(selected, indent=2))

{
  "selected_on": "validation period, 2022-01-03 to 2023-12-29",
  "criterion": "lowest validation RMSE",
  "arima": {
    "p": 1,
    "d": 0,
    "q": 0
  },
  "random_forest": {
    "target": "change",
    "features": "har",
    "min_samples_leaf": 50,
    "split_features": "third"
  }
}


In [9]:
summary = pd.concat([
    reference,
    pd.DataFrame({
        "ARIMA({p},{d},{q})".format(**selected["arima"]): best_arima[["RMSE", "MAE"]],
        "Random Forest (selected)": best_forest[["RMSE", "MAE"]],
    }).T,
]).astype(float)
summary["RMSE / naive"] = summary["RMSE"] / summary.loc["Naive", "RMSE"]
summary["MAE / naive"] = summary["MAE"] / summary.loc["Naive", "MAE"]
summary.round(4)

,RMSE,MAE,RMSE / naive,MAE / naive
Naive,1.4279,1.0335,1.0000,1.0000
HAR,1.4220,1.0105,0.9958,0.9778
"ARIMA(1,0,0)",1.4221,1.0124,0.9959,0.9796
Random Forest (selected),1.4193,1.0214,0.9940,0.9883


**Reading.** On the validation period the four forecasts are almost indistinguishable: their RMSE values span 1.4193 to 1.4279, a range of 0.6%. On MAE the HAR regression is lowest, 2.2% below naive.

Two cautions before the test evaluation:

- The selected ARIMA and Random Forest candidates were chosen *because* they had the lowest validation error among 18 and 24 candidates. Their validation errors are therefore optimistic, and margins this small could easily disappear on new data.
- No significance test is run here. The protocol reserves testing for the test period, where the selection is already fixed.

## 7. Save the selection

The selected configurations are written to `results/selected_models.json`, which the test-evaluation notebook reads. The validation errors of every candidate are written to `results/validation_candidates.csv`.

In [10]:
(RESULTS / "selected_models.json").write_text(json.dumps(selected, indent=2) + "\n", encoding="utf-8")

candidates = pd.concat([
    arima_table.assign(family="ARIMA", candidate=[f"ARIMA({r.p},{r.d},{r.q})" for r in arima_table.itertuples()]),
    forest_table.assign(family="Random Forest", candidate=[
        f"RF({r.target}, {r.features}, leaf={r.min_samples_leaf}, split={r.split_features})"
        for r in forest_table.itertuples()
    ]),
    reference.assign(family="Reference", candidate=reference.index).assign(
        **{"RMSE / naive": reference["RMSE"] / naive_rmse}
    ),
])
columns = ["family", "candidate", "RMSE", "MAE", "RMSE / naive"]
candidates[columns].round(6).to_csv(RESULTS / "validation_candidates.csv", index=False, lineterminator="\n")
print(f"Saved {len(candidates)} rows")

Saved 44 rows
